In [1]:
import pandas as pd
import os
import capnp
import glob
import multiprocessing
from collections import defaultdict
from scipy.spatial import cKDTree

In [9]:
# prelim frequency reduction, remove files with frequencies all less than 2 GHz


directory = "/datax/scratch/jaym/meerkat_data/"

# Get list of candidate files
folders = [f for f in os.listdir(directory) if f.startswith("blpn")]
full_paths = [os.path.join(directory, f) for f in folders]

hits_paths=[]
for path in full_paths:
    for subfolder in os.listdir(path):
        subfolder_path = os.path.join(path, subfolder)
        if os.path.isdir(subfolder_path):  
            for file in os.listdir(subfolder_path):
                if file.endswith(".hits"):
                   hits_paths.append(os.path.join(subfolder_path, file))

In [10]:
print(len(hits_paths))
print(hits_paths[0])

252
/datax/scratch/jaym/meerkat_data/blpn0_scratch_data_20240319T020710Z-20240219-0019/seticore_search/guppi_60388_07630_014132_J1658-5324_0001.hits


In [13]:
# Load your local hit.capnp schema
hit_capnp = capnp.load("/home/ellambishop/seticore/hit.capnp")

def hits_to_df(hits, exclude_data=True):
    data = []
    for h in hits:
        entry = {}
        # fields is a dict with field names as keys
        for fname in h.schema.as_struct().fields.keys():
            value = getattr(h, fname)
            if hasattr(value, "schema"):  # nested struct
                # Get subfield names as dict keys as well
                subfields = value.schema.fields
                if isinstance(subfields, dict):
                    subfield_names = subfields.keys()
                else:
                    # fallback: list or other
                    try:
                        subfield_names = [sf.name for sf in subfields]
                    except Exception:
                        subfield_names = subfields

                for sfname in subfield_names:
                    if exclude_data and sfname == "data":
                        continue
                    entry[f"{fname}_{sfname}"] = getattr(value, sfname)
            else:
                entry[fname] = value
        data.append(entry)
    return pd.DataFrame(data)


pickle_dir = "/datax/scratch/ellambishop/meerkat_pkls/"
os.makedirs(pickle_dir, exist_ok=True)  # Create directory if it doesn't exist

for idx, hits_file in enumerate(hits_paths):
    with open(hits_file, "rb") as f:
        hits = list(hit_capnp.Hit.read_multiple(f))
    
    df = hits_to_df(hits)

    # Create unique pickle filename
    base_name = os.path.splitext(os.path.basename(hits_file))[0]
    pickle_name = f"{idx:03d}_seticore_search_{base_name}.pkl"
    pickle_path = os.path.join(pickle_dir, pickle_name)

    df.to_pickle(pickle_path)
    print(f"Saved {pickle_path}")


Saved /datax/scratch/ellambishop/meerkat_pkls/000_seticore_search_guppi_60388_07630_014132_J1658-5324_0001.pkl
Saved /datax/scratch/ellambishop/meerkat_pkls/001_seticore_search_guppi_60578_82556_000102_J0042+1246_0001.pkl
Saved /datax/scratch/ellambishop/meerkat_pkls/002_seticore_search_guppi_60578_83894_000108_J0042+1246_0001.pkl
Saved /datax/scratch/ellambishop/meerkat_pkls/003_seticore_search_guppi_60579_00171_000118_J0042+1246_0001.pkl
Saved /datax/scratch/ellambishop/meerkat_pkls/004_seticore_search_guppi_60578_82556_000102_J0042+1246_0001.pkl
Saved /datax/scratch/ellambishop/meerkat_pkls/005_seticore_search_guppi_60578_83894_000108_J0042+1246_0001.pkl
Saved /datax/scratch/ellambishop/meerkat_pkls/006_seticore_search_guppi_60579_00171_000118_J0042+1246_0001.pkl
Saved /datax/scratch/ellambishop/meerkat_pkls/007_seticore_search_guppi_60578_82556_000102_J0042+1246_0001.pkl
Saved /datax/scratch/ellambishop/meerkat_pkls/008_seticore_search_guppi_60578_83894_000108_J0042+1246_0001.pkl
S

In [24]:
df = pd.read_pickle('/datax/scratch/ellambishop/meerkat_pkls/028_seticore_search_guppi_60578_82556_000102_J0042+1246_0001.pkl')
df.columns

Index(['signal_frequency', 'signal_index', 'signal_driftSteps',
       'signal_driftRate', 'signal_snr', 'signal_coarseChannel', 'signal_beam',
       'signal_numTimesteps', 'signal_power', 'signal_incoherentPower',
       'filterbank_sourceName', 'filterbank_fch1', 'filterbank_foff',
       'filterbank_tstart', 'filterbank_tsamp', 'filterbank_ra',
       'filterbank_dec', 'filterbank_telescopeId', 'filterbank_numTimesteps',
       'filterbank_numChannels', 'filterbank_coarseChannel',
       'filterbank_startChannel', 'filterbank_beam'],
      dtype='object')

In [ ]:
def reduction(df):
    # Identify coherent (phase center) vs incoherent beams
    df = df[(df['signal_drift_rate'] != 0)]
    df['is_coherent'] = df['signal_beam'] == df['filterbank_beam']

    df['signal_duration'] = df['signal_num_timesteps'] * df['filterbank_tsamp']

    cond1 = (
        (df['signal_duration'] >= 20) &  # e.g. 20–80 seconds
        (df['signal_duration'] <= 80) &
        (df['signal_snr'] > 10)
    )
    cond2 = (
        (df['signal_duration'] < 20) &   # short impulsive RFI
        (df['signal_snr'] > 15)
    )

    drop_mask = cond1 | cond2
    df = df.loc[~drop_mask]

    # Split
    coh_df = df[df['is_coherent']].copy()        # Coherent (on-target beam)
    incoh_df = df[~df['is_coherent']].copy()     # Incoherent (off beams)

    return coh_df, incoh_df
